# Session 5 — NumPy, and the Signal We Will Keep

**Goal of this session:** replace Python lists with NumPy arrays, and build the toy signal that carries us through session 9.

*Python for Neuroscience, session 5 of 12.*

## Why this matters

A one-minute EEG recording at 500 Hz is 30,000 numbers per channel. A Python list will hold them, but every operation means a loop, and loops in pure Python are slow enough to hurt.

NumPy stores numbers in a compact block and applies one operation to the whole block at once. That idea, called vectorisation, is the foundation of every scientific package you will use later.

## Lists are slow. Let's prove it.

Same job twice: multiply a million numbers by two.

In [ ]:
import time
import numpy as np

n = 1_000_000
py_list = list(range(n))
np_array = np.arange(n)

t0 = time.time()
doubled_list = [x * 2 for x in py_list]
list_time = time.time() - t0

t0 = time.time()
doubled_array = np_array * 2
array_time = time.time() - t0

print(f"list:  {list_time * 1000:7.1f} ms")
print(f"array: {array_time * 1000:7.1f} ms")
print(f"NumPy was about {list_time / array_time:.0f}x faster")

Notice what the array version looks like. No loop, no brackets, just `np_array * 2`. That is the style you want to get used to.

## Making arrays

There are a few ways in, and you will use all of them.

In [ ]:
import numpy as np

a = np.array([1, 2, 3, 4, 5])          # from a list
b = np.zeros(5)                         # five zeros
c = np.arange(0, 1, 0.2)                # start, stop, step
d = np.linspace(0, 1, 5)                # five values from 0 to 1 inclusive

print(a)
print(b)
print(c)
print(d)
print(a.shape, a.dtype)

`arange` and `linspace` look similar and are not. `arange` takes a step size and excludes the endpoint. `linspace` takes a count and includes it. For time axes you almost always want `arange`, because you know your sampling interval.

## Our running signal

Here is the function we will re-use for the next five sessions. It builds a sine wave at 10 Hz, which is in the alpha range, and adds white noise on top.

Be clear with yourself about what this is: a toy. It is not a recording of anything. It exists so that we can practise on something that behaves like a brain signal without pretending it is one.

In [ ]:
import numpy as np


def generate_toy_signal(duration=2.0, sampling_rate=500.0, noise_level=0.5,
                        freq=10.0, amplitude=1.0, seed=0):
    """A toy oscillatory signal: one sine wave plus white noise.

    This is not a recording. It is a stand-in that behaves enough like an
    alpha rhythm to practise on. Returns the time axis and the signal.
    """
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / sampling_rate)
    signal = amplitude * np.sin(2 * np.pi * freq * t)
    signal = signal + noise_level * rng.standard_normal(t.size)
    return t, signal

In [ ]:
t, signal = generate_toy_signal(duration=2.0, sampling_rate=500.0, noise_level=0.5)

print("time axis:", t.shape, "from", t[0], "to", t[-1], "s")
print("signal:   ", signal.shape, signal.dtype)
print("first five values:", signal[:5])

Read the function once more. `2 * np.pi * freq * t` is the whole time axis multiplied through in one go, and `np.sin()` of that is the whole sine wave. No loop anywhere.

## Looking at it

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, signal, linewidth=1, color="#2b6cb0")
ax.set_xlabel("time (s)", fontsize=13)
ax.set_ylabel("amplitude", fontsize=13)
ax.set_title("Toy oscillatory signal, 10 Hz plus noise", fontsize=15)
ax.tick_params(labelsize=12)
plt.tight_layout()
plt.show()

## Describing an array

NumPy has the summary statistics built in as methods on the array itself.

In [ ]:
print(f"mean      {signal.mean():7.3f}")
print(f"std       {signal.std():7.3f}")
print(f"min       {signal.min():7.3f}")
print(f"max       {signal.max():7.3f}")
print(f"peak time {t[signal.argmax()]:7.3f} s")

`argmax` gives the *position* of the maximum rather than the value, and feeding that position into the time axis tells you when it happened. That two-step move is worth remembering.

## Slicing a time window

Arrays slice exactly like lists, but the useful trick is a boolean mask: write the condition you want, and NumPy hands back only the matching values.

In [ ]:
# first 200 samples
print(signal[:200].shape)

# everything between 0.5 and 1.0 seconds, by condition
window = (t >= 0.5) & (t < 1.0)
print(window[:10])                     # a mask of True and False
print(signal[window].shape, "samples in the window")
print(f"mean in window: {signal[window].mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t, signal, linewidth=1, color="#cbd5e0", label="full signal")
ax.plot(t[window], signal[window], linewidth=1.5, color="#2b6cb0", label="0.5 to 1.0 s")
ax.set_xlabel("time (s)", fontsize=13)
ax.set_ylabel("amplitude", fontsize=13)
ax.set_title("Selecting a time window with a mask", fontsize=15)
ax.tick_params(labelsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## Try it yourself

Call `generate_toy_signal(noise_level=2.0)` and plot it. The oscillation is still there mathematically, but you will struggle to see it. Session 8 is about getting it back.

**Next session:** pandas, for the trial tables and behavioural data that sit alongside your signals.